# RT-BENE eye-state training on Kaggle
Open this local notebook in VS Code and select the **Kaggle Jupyter Server** as its kernel. Run cells in order. The large dataset stays on Kaggle; only model results are saved under `/kaggle/working/artifacts`. This notebook uses a pinned GitHub commit so a run can be reproduced.

In [ ]:
from pathlib import Path
import subprocess, sys, torch
assert Path('/kaggle/working').is_dir(), 'Select the Kaggle remote Jupyter server in VS Code first.'
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'unavailable')
assert torch.cuda.is_available(), 'Enable GPU in Kaggle Settings > Accelerator, then reconnect.'

## Fetch the project code
GitHub holds the code; the VS Code notebook controls execution. Change `SOURCE_COMMIT` only when you intentionally want a different code version.

In [ ]:
SOURCE_COMMIT = '5aa2df8fcfa19facf362002e232a95f95b47d637'
repo = Path('/kaggle/temp/driver-monitor-code')
if not (repo / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/mfahadwaqar/DLCV-Driver-Distraction-Detection.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
(repo / 'SOURCE_COMMIT').write_text(SOURCE_COMMIT + '\n')
print('Project code:', repo, SOURCE_COMMIT)

## Fetch and prepare RT-BENE
The official Zenodo files are checked against their declared sizes and MD5 hashes. Unlabeled and disputed frames are excluded. The prepared images stay in Kaggle's temporary storage.

In [ ]:
data = Path('/kaggle/temp/rt-bene')
subprocess.run([sys.executable, '-m', 'src.data.fetch_rt_bene', '--output', str(data / 'source')], cwd=repo, check=True)
subprocess.run([sys.executable, '-m', 'src.data.rt_bene', '--source', str(data / 'source'), '--output', str(data / 'prepared')], cwd=repo, check=True)
print('Manifest:', data / 'prepared' / 'manifest.csv')

## Train and evaluate
Set `RUN_TRAINING` to `True` when ready. This is the full five-epoch baseline, using subject-separated train, validation, and test splits. Each output folder must be new so earlier results are preserved.

In [ ]:
RUN_TRAINING = False
RUN_NAME = 'rt-bene-full-01'
output = Path('/kaggle/working/artifacts') / RUN_NAME
splits = Path('/kaggle/working/artifacts/rt-bene-splits.json')
if RUN_TRAINING:
    subprocess.run([sys.executable, '-m', 'src.train.phase1', '--data-root', str(data / 'prepared'), '--manifest', str(data / 'prepared' / 'manifest.csv'), '--output', str(output), '--splits', str(splits), '--device', 'cuda'], cwd=repo, check=True)
else:
    print('Set RUN_TRAINING = True to start full training.')

In [ ]:
import json
if (output / 'summary.json').exists():
    print(json.dumps(json.loads((output / 'summary.json').read_text()), indent=2))
else:
    print('No completed run in', output)